prompt 

Write SQL to build a silver layer from small_projects.finops.bronze_billing_usage:

1. Create small_projects.finops.silver_billing_usage:
   - remove duplicate rows
   - cast usage_date to DATE; cast uptime_hours, active_hours and dbus to DOUBLE
   - convert workspace_name to lowercase
   - replace NULL or empty tag_team with 'untagged'
   - keep only rows where active_hours <= uptime_hours
   - drop the _rescued_data column

2. Create small_projects.finops.quarantine_billing_usage with the rows where
   active_hours > uptime_hours (duplicates removed, without _rescued_data).

3. Create small_projects.finops.silver_resources and small_projects.finops.silver_sku_prices
   as copies of the bronze tables without _rescued_data.

4. Add a check query that returns: row count of silver_billing_usage,
   row count of quarantine_billing_usage, and the number of distinct workspace_name values in silver. keep all original columns in both the silver and quarantine tables,
including record_id, resource_id, sku_name and job_name

In [0]:
%sql
-- ------------------------------------------------------------------
-- Silver layer for finops billing usage (from bronze, cleaned + split)
-- ------------------------------------------------------------------

-- Silver: deduplicated, typed, normalized rows that pass the sanity check
CREATE OR REPLACE TABLE small_projects.finops.silver_billing_usage AS
SELECT DISTINCT
  record_id,
  CAST(usage_date AS DATE)                   AS usage_date,
  CAST(uptime_hours AS DOUBLE)               AS uptime_hours,
  CAST(active_hours AS DOUBLE)             AS activate_hours,
  CAST(dbus AS DOUBLE)                       AS dbus,
  LOWER(workspace_name)                      AS workspace_name,
  COALESCE(NULLIF(tag_team, ''), 'untagged') AS tag_team,
  resource_id,
  sku_name,
  job_name
FROM small_projects.finops.bronze_billing_usage
WHERE active_hours <= uptime_hours;

-- Quarantine: rows violating activate_hours <= uptime_hours (same cleanup, no _rescued_data)
CREATE OR REPLACE TABLE small_projects.finops.quarantine_billing_usage AS
SELECT DISTINCT
  record_id,
  CAST(usage_date AS DATE)                   AS usage_date,
  CAST(uptime_hours AS DOUBLE)               AS uptime_hours,
  CAST(active_hours AS DOUBLE)             AS activate_hours,
  CAST(dbus AS DOUBLE)                       AS dbus,
  LOWER(workspace_name)                      AS workspace_name,
  COALESCE(NULLIF(tag_team, ''), 'untagged') AS tag_team,
  resource_id,
  sku_name,
  job_name
FROM small_projects.finops.bronze_billing_usage
WHERE active_hours > uptime_hours;

-- Silver copies of the other bronze tables (drop _rescued_data)
CREATE OR REPLACE TABLE small_projects.finops.silver_resources AS
SELECT * EXCEPT (_rescued_data)
FROM small_projects.finops.bronze_resources;

CREATE OR REPLACE TABLE small_projects.finops.silver_sku_prices AS
SELECT * EXCEPT (_rescued_data)
FROM small_projects.finops.bronze_sku_prices;

-- ------------------------------------------------------------------
-- Check: row counts and distinct workspaces
-- ------------------------------------------------------------------
SELECT
  (SELECT COUNT(*) FROM small_projects.finops.silver_billing_usage)     AS silver_billing_rows,
  (SELECT COUNT(*) FROM small_projects.finops.quarantine_billing_usage) AS quarantine_billing_rows,
  (SELECT COUNT(DISTINCT workspace_name)
     FROM small_projects.finops.silver_billing_usage)                    AS distinct_workspace_names;
